# SPIMquant analysis tutorial

This notebook walks through a lightweight SPIMquant analysis workflow using the public workshop archive:

- download or reuse the public workshop dataset
- inspect the directory layout and key SPIMquant outputs
- review subject-level and cohort-level regional statistics
- identify QC artefacts to open in external viewers

The notebook is designed to stay useful even when optional plotting packages are unavailable.


## Prerequisites

- A local Jupyter environment
- Access to the public workshop archive: `https://object-arbutus.alliancecan.ca/a8337d65c739485795cd19794e1fffa6:mind-public/workshop-data/spimquant_workshop_exclude_nifti_zarr.zip`
- Optional packages for richer analysis: `pandas` and `matplotlib`

By default, the notebook downloads and extracts the archive under `/tmp/spimquant_workshop` so the repository checkout stays clean.


In [ ]:
from __future__ import annotations

import csv
import importlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

DATA_URL = 'https://object-arbutus.alliancecan.ca/a8337d65c739485795cd19794e1fffa6:mind-public/workshop-data/spimquant_workshop_exclude_nifti_zarr.zip'
WORK_DIR = Path('/tmp/spimquant_workshop')
ARCHIVE_PATH = WORK_DIR / 'spimquant_workshop_exclude_nifti_zarr.zip'
EXTRACT_DIR = WORK_DIR / 'spimquant_workshop_exclude_nifti_zarr'
WORK_DIR.mkdir(parents=True, exist_ok=True)


def optional_import(name: str):
    try:
        return importlib.import_module(name)
    except Exception as exc:
        print(f"Optional package {name!r} is unavailable: {exc}")
        return None


pd = optional_import('pandas')
plt = optional_import('matplotlib.pyplot')
if plt is not None:
    try:
        plt.style.use('seaborn-v0_8-whitegrid')
    except Exception:
        pass


def download_archive(url: str, destination: Path) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url) as response, destination.open('wb') as fout:
            shutil.copyfileobj(response, fout)
        return True
    except Exception as exc:
        print(f'Download failed: {exc}')
        print('If needed, download the archive manually and place it here:')
        print(destination)
        return False


def ensure_dataset() -> Path | None:
    if EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
        print(f'Using extracted data in {EXTRACT_DIR}')
    else:
        if not ARCHIVE_PATH.exists():
            print(f'Downloading workshop archive to {ARCHIVE_PATH}')
            if not download_archive(DATA_URL, ARCHIVE_PATH):
                return None
        print(f'Extracting archive into {EXTRACT_DIR}')
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ARCHIVE_PATH) as zf:
            zf.extractall(EXTRACT_DIR)

    visible_children = [p for p in EXTRACT_DIR.iterdir() if not p.name.startswith('.')]
    if len(visible_children) == 1 and visible_children[0].is_dir():
        return visible_children[0]
    return EXTRACT_DIR


dataset_root = ensure_dataset()
dataset_root


In [ ]:
from collections import defaultdict


def describe_tree(root: Path, max_depth: int = 2, max_entries: int = 80):
    if root is None:
        print('Dataset is not available in this environment.')
        return

    rows = []
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        prefix = '  ' * (depth - 1)
        suffix = '/' if path.is_dir() else ''
        rows.append(f'{prefix}{path.name}{suffix}')
        if len(rows) >= max_entries:
            rows.append('...')
            break
    print('\n'.join(rows))


def find_paths(root: Path | None, pattern: str):
    return sorted(root.rglob(pattern)) if root is not None else []


def read_tsv_records(path: Path, limit: int | None = None):
    with path.open(newline='') as fin:
        reader = csv.DictReader(fin, delimiter='	')
        rows = list(reader)
    if limit is not None:
        rows = rows[:limit]
    return rows


def preview_table(path: Path, limit: int = 5):
    records = read_tsv_records(path, limit=limit)
    if pd is not None:
        return pd.DataFrame(records)
    for row in records:
        print(row)
    return records


def choose_metric_column(columns):
    priorities = (
        'density',
        'fieldfrac',
        'count',
        'nvoxels',
        'cohens',
        'tstat',
        'pvalue',
        'p_val',
        'pvalue_adj',
    )
    lowered = {col.lower(): col for col in columns}
    for key in priorities:
        for lower_name, original_name in lowered.items():
            if key in lower_name:
                return original_name
    return columns[0] if columns else None


def is_pvalue_like(column: str) -> bool:
    lower = column.lower()
    exact_names = {
        'p',
        'pval',
        'pvalue',
        'p_value',
        'adj_p',
        'adj_pval',
        'adj_pvalue',
        'padj',
        'fdr',
        'qvalue',
        'q_value',
    }
    suffixes = (
        '_p',
        '_pval',
        '_pvalue',
        '_p_value',
        '_adj_p',
        '_adj_pval',
        '_adj_pvalue',
        '_padj',
        '_fdr',
        '_qvalue',
        '_q_value',
    )
    return lower in exact_names or lower.endswith(suffixes)


def numeric_columns(records):
    if not records:
        return []
    cols = []
    for key in records[0]:
        saw_numeric = False
        try:
            for row in records[:10]:
                value = row.get(key, '')
                if value in ('', None):
                    continue
                float(value)
                saw_numeric = True
            if saw_numeric:
                cols.append(key)
        except Exception:
            continue
    return cols


def region_column(records):
    if not records:
        return None
    for name in records[0]:
        lower = name.lower()
        if 'region' in lower or 'label' in lower or 'name' in lower:
            return name
    return next(iter(records[0]), None)


## Explore the extracted dataset

The workshop archive can contain BIDS metadata, SPIMquant outputs, or both. Start by checking what was packaged.


In [ ]:
describe_tree(dataset_root, max_depth=2, max_entries=120)


## Locate the most useful analysis files

The tutorial looks for a few common artefacts automatically:

- `participants.tsv` for cohort metadata
- `*_mergedsegstats.tsv` for subject-level regional measurements
- `*_groupstats.tsv` for group-level statistics
- registration QC HTML and QC PNGs for manual review


In [ ]:
participants_files = find_paths(dataset_root, 'participants.tsv')
subject_stats_files = [
    path for path in find_paths(dataset_root, '*.tsv')
    if 'mergedsegstats' in path.name.lower()
]
group_stats_files = [
    path for path in find_paths(dataset_root, '*.tsv')
    if 'groupstats' in path.name.lower()
]
regqc_files = [path for path in find_paths(dataset_root, '*.html') if 'regqc' in path.name.lower()]
qc_png_files = [path for path in find_paths(dataset_root, '*.png') if 'qc' in str(path).lower()]

print('participants.tsv files:', len(participants_files))
print('subject stats files:', len(subject_stats_files))
print('group stats files:', len(group_stats_files))
print('registration QC html files:', len(regqc_files))
print('QC png files:', len(qc_png_files))

for label, files in {
    'participants': participants_files,
    'subject stats': subject_stats_files[:3],
    'group stats': group_stats_files[:3],
    'registration QC': regqc_files[:3],
    'QC PNG': qc_png_files[:3],
}.items():
    if files:
        print(f'\nExample {label} paths:')
        for path in files:
            print(' -', path)


## Preview participant metadata

If the archive includes a `participants.tsv`, inspect it first because SPIMquant group analyses usually merge this metadata back into the output tables.


In [ ]:
if participants_files:
    display(preview_table(participants_files[0], limit=10))
else:
    print('No participants.tsv file was found in the extracted archive.')


## Review one subject-level regional table

A `*_mergedsegstats.tsv` file usually contains one row per region with quantitative measures such as field fraction, density, or counts.


In [ ]:
if subject_stats_files:
    subject_records = read_tsv_records(subject_stats_files[0])
    print(f'Loaded {len(subject_records)} rows from {subject_stats_files[0].name}')
    metric_columns = numeric_columns(subject_records)
    region_name_column = region_column(subject_records)
    chosen_metric = choose_metric_column(metric_columns)
    print('Region column:', region_name_column)
    print('Candidate numeric columns:', metric_columns[:10])
    print('Chosen metric column:', chosen_metric)
    display(preview_table(subject_stats_files[0], limit=5))
else:
    print('No subject-level mergedsegstats table was found.')


In [ ]:
if subject_stats_files:
    subject_records = read_tsv_records(subject_stats_files[0])
    metric_columns = numeric_columns(subject_records)
    region_name_column = region_column(subject_records)
    chosen_metric = choose_metric_column(metric_columns)

    if chosen_metric is None or region_name_column is None:
        print('Could not identify a region column and numeric metric column automatically.')
    else:
        ranked = []
        for row in subject_records:
            value = row.get(chosen_metric, '')
            try:
                ranked.append((float(value), row.get(region_name_column, 'unknown')))
            except Exception:
                continue
        ranked.sort(reverse=True)
        top_regions = ranked[:10]
        print(f'Top 10 regions by {chosen_metric}:')
        for value, name in top_regions:
            print(f' - {name}: {value:.4f}')

        if pd is not None:
            frame = pd.DataFrame([
                {'region': name, chosen_metric: value}
                for value, name in top_regions
            ])
            display(frame)
            if plt is not None and not frame.empty:
                ax = frame.iloc[::-1].plot.barh(x='region', y=chosen_metric, figsize=(8, 5), legend=False)
                ax.set_title(f'Top regions in {subject_stats_files[0].name}')
                ax.set_xlabel(chosen_metric)
                plt.tight_layout()
                plt.show()
else:
    print('Skipping regional ranking because no subject-level table was found.')


## Next step: run or adapt SPIMquant itself

After reviewing the packaged outputs, you can run a fresh SPIMquant analysis on your own BIDS dataset with the same concepts used above.

```bash
# Dry run
pixi run spimquant /path/to/bids /path/to/output participant -np

# Faster tutorial-style participant run
pixi run spimquant /path/to/bids /path/to/output participant \
  --segmentation-level 3 \
  --cores all

# Group analysis
pixi run spimquant /path/to/bids /path/to/output group \
  --contrast-column treatment \
  --contrast-values control drug \
  --cores all
```


In [ ]:
if len(subject_stats_files) > 1:
    aggregated = defaultdict(list)
    chosen_metric = None
    region_name_column = None

    for path in subject_stats_files:
        records = read_tsv_records(path)
        metric_columns = numeric_columns(records)
        region_name_column = region_name_column or region_column(records)
        chosen_metric = chosen_metric or choose_metric_column(metric_columns)
        if chosen_metric is None or region_name_column is None:
            continue
        for row in records:
            try:
                aggregated[row[region_name_column]].append(float(row[chosen_metric]))
            except Exception:
                continue

    cohort_rows = [
        {
            'region': region,
            'n_subjects': len(values),
            f'mean_{chosen_metric}': sum(values) / len(values),
        }
        for region, values in aggregated.items()
        if values
    ]
    cohort_rows.sort(key=lambda row: row[f'mean_{chosen_metric}'], reverse=True)

    if pd is not None:
        cohort_df = pd.DataFrame(cohort_rows[:10])
        display(cohort_df)
        if plt is not None and not cohort_df.empty:
            ax = cohort_df.iloc[::-1].plot.barh(x='region', y=f'mean_{chosen_metric}', figsize=(8, 5), legend=False)
            ax.set_title(f'Cohort mean by region ({chosen_metric})')
            ax.set_xlabel(f'mean_{chosen_metric}')
            plt.tight_layout()
            plt.show()
    else:
        for row in cohort_rows[:10]:
            print(row)
else:
    print('A cohort summary needs at least two subject-level regional tables.')


## Inspect group-level statistics

When SPIMquant `group` outputs are present, sort them by a likely significance column to find the strongest regional effects quickly.


In [ ]:
if group_stats_files:
    group_records = read_tsv_records(group_stats_files[0])
    metric_columns = numeric_columns(group_records)
    region_name_column = region_column(group_records)
    p_like_columns = [col for col in metric_columns if is_pvalue_like(col)]
    chosen_metric = p_like_columns[0] if p_like_columns else choose_metric_column(metric_columns)
    print('Region column:', region_name_column)
    print('Chosen ranking column:', chosen_metric)

    if chosen_metric is None:
        print('No numeric group-level columns were detected.')
    else:
        ascending = is_pvalue_like(chosen_metric)
        ranked = []
        for row in group_records:
            try:
                ranked.append((float(row[chosen_metric]), row))
            except Exception:
                continue
        ranked.sort(reverse=not ascending)
        top_rows = [row for _, row in ranked[:10]]
        if pd is not None:
            display(pd.DataFrame(top_rows))
        else:
            for row in top_rows:
                print(row)
else:
    print('No group-level stats table was found in the archive.')


## QC artefacts to open outside the notebook

Notebook summaries are helpful, but final QC should still happen in viewers such as a web browser, ITK-SNAP, or napari.


In [ ]:
for title, files in [
    ('Registration QC HTML', regqc_files),
    ('QC PNG overlays', qc_png_files),
]:
    print(f'\n{title}:')
    if files:
        for path in files[:10]:
            print(' -', path)
    else:
        print(' - none found')


## Next step: run or adapt SPIMquant itself

After reviewing the packaged outputs, you can run a fresh SPIMquant analysis on your own BIDS dataset with the same concepts used above.

```bash
# Dry run
pixi run spimquant /path/to/bids /path/to/output participant -np

# Faster tutorial-style participant run
pixi run spimquant /path/to/bids /path/to/output participant   --segmentation-level 3   --cores all

# Group analysis
pixi run spimquant /path/to/bids /path/to/output group   --contrast-column treatment   --contrast-values control drug   --cores all
```
